# 04. Model Selection and Comparison

Rank candidate distributions from BestFit likelihoods and compute Akaike-weighted model-average quantiles.

In [ ]:
import pythonnet
pythonnet.load("netfx")

import clr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from helper_functions import resolve_bestfit_dll, resolve_numerics_dll, convert_to_donet_array

clr.AddReference(str(resolve_numerics_dll()))
clr.AddReference(str(resolve_bestfit_dll()))

from RMC.BestFit import DataFrame, ExactData
from RMC.BestFit.Model import UnivariateDistribution
from Numerics.Distributions import UnivariateDistributionType

In [ ]:
rng = np.random.default_rng(9)
years = np.arange(1970, 1970 + 60)
peaks = np.maximum(500, 9300 + 32*(years-years.min()) + rng.normal(0, 920, len(years)))

df = DataFrame(); [df.ExactSeries.Add(ExactData(int(y), float(q))) for y, q in zip(years, peaks)]
x = np.array(peaks, dtype=float)
mu, sd = float(np.mean(x)), float(np.std(x, ddof=1))

specs = {
    'Normal': (UnivariateDistributionType.Normal, [mu, sd]),
    'Gumbel': (UnivariateDistributionType.Gumbel, [mu, 0.80*sd]),
    'GEV': (UnivariateDistributionType.GeneralizedExtremeValue, [mu, 0.80*sd, 0.08]),
}

def robust_loglik(model, p_arr, data):
    try:
        ll=float(model.LogLikelihood(p_arr))
        if np.isfinite(ll) and ll > -1e300:
            return ll
    except Exception:
        pass
    pdf=np.array([max(1e-300,float(model.Distribution.PDF(float(v)))) for v in data],dtype=float)
    return float(np.sum(np.log(pdf)))

rows=[]; models={}
for name,(dtype,pvals) in specs.items():
    m=UnivariateDistribution(df,dtype)
    p=convert_to_donet_array(pvals)
    m.SetParameterValues(p)
    ll=robust_loglik(m,p,x)
    k=len(pvals)
    rows.append({'model':name,'k':k,'log_likelihood':ll,'AIC':2*k-2*ll})
    models[name]=m

summary=pd.DataFrame(rows).sort_values('AIC')
summary['delta_AIC']=summary['AIC']-summary['AIC'].min()
summary['akaike_weight']=np.exp(-0.5*summary['delta_AIC'])
summary['akaike_weight']=summary['akaike_weight']/summary['akaike_weight'].sum()
summary

In [ ]:
weights=dict(zip(summary['model'], summary['akaike_weight']))
T=np.array([2,5,10,25,50,100],dtype=float)
F=1.0-1.0/T
rows=[]
for t,f in zip(T,F):
    row={'return_period':float(t)}
    qbar=0.0
    for name,m in models.items():
        q=float(m.Distribution.InverseCDF(float(f)))
        row[name]=q
        qbar += weights[name]*q
    row['model_avg']=qbar
    rows.append(row)
qtab=pd.DataFrame(rows)
qtab

In [ ]:
plt.figure(figsize=(8,5))
for c in ['Normal','Gumbel','GEV','model_avg']:
    plt.plot(qtab['return_period'], qtab[c], marker='o', lw=3 if c=='model_avg' else 2, label=c)
plt.xscale('log')
plt.xlabel('Return period')
plt.ylabel('Quantile')
plt.title('BestFit Model Comparison + Model Averaging')
plt.grid(alpha=0.3, which='both')
plt.legend()
plt.tight_layout()

## Summary and Exercises

You used BestFit likelihood outputs to perform model ranking and weighted quantile aggregation.

Exercises:
1. Add BIC-based weights and compare with AIC weights.
2. Run at multiple sample sizes to assess rank stability.
3. Add uncertainty fan plots around model-average curve.